# 4D \(SU(2)\) validated RG — GPU session driver

This notebook is the primary execution interface for the design in
`validated_4d_su2_rg_codex_design.md`.

**Hard runtime rule:** checkpoint to persistent storage and return within
5 hours 30 minutes. Notebook RAM is never treated as durable state.

The current scaffold implements M0 persistence/session control. Mathematical RG
phases remain `NOT_CERTIFIED` until explicitly implemented and validated.

## Startup procedure

1. Store this notebook itself in persistent storage.
2. Mount a persistent volume or Google Drive.
3. Set `VALIDATED_RG_PERSIST_ROOT` to a directory that survives runtime shutdown.
4. Run the cells from the top.
5. Call `orchestrator.run_until_checkpoint()`.

After shutdown, repeat the mount/bootstrap steps. The loader verifies hashes and
resumes from the newest valid committed checkpoint.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import sys
import time
import uuid
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Literal

import numpy as np

try:
    import torch
except ImportError:
    torch = None

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Torch:', getattr(torch, '__version__', None))
print('CUDA available:', bool(torch and torch.cuda.is_available()))
if torch and torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('GPU:', torch.cuda.get_device_name(0))

## Persistent root

Do not use `/tmp` or runtime-local storage. Example for Colab after mounting
Drive:

```python
os.environ['VALIDATED_RG_PERSIST_ROOT'] = \
    '/content/drive/MyDrive/validated_rg'
```


In [ ]:
PERSIST_ROOT_ENV = 'VALIDATED_RG_PERSIST_ROOT'

def resolve_persistent_root() -> Path:
    raw = os.environ.get(PERSIST_ROOT_ENV)
    if not raw:
        raise RuntimeError(
            f'Set {PERSIST_ROOT_ENV} to a directory that survives runtime shutdown.'
        )
    root = Path(raw).expanduser().resolve()
    if root == Path('/tmp') or Path('/tmp') in root.parents:
        raise RuntimeError('/tmp is not accepted as persistent storage.')
    if root == Path('/content'):
        raise RuntimeError('/content root is normally ephemeral; choose a mounted path.')
    root.mkdir(parents=True, exist_ok=True)
    probe = root / '.persistence_write_probe'
    probe.write_text(datetime.now(timezone.utc).isoformat(), encoding='utf-8')
    if not probe.exists():
        raise RuntimeError('Persistent-root write probe failed.')
    return root

# PERSIST_ROOT = resolve_persistent_root()

## Immutable configuration and run identity

In [ ]:
@dataclass(frozen=True)
class RunConfig:
    schema_version: int = 1
    project_name: str = 'validated_4d_su2_rg'
    beta_num: int = 11
    beta_den: int = 5
    j2_max: int = 1
    bond_dim: int = 8
    oversample: int = 4
    power_iterations: int = 1
    rg_steps: int = 1
    seed: int = 20260719
    checkpoint_interval_s: int = 15 * 60
    max_work_item_s: int = 20 * 60
    no_long_task_after_s: int = 5 * 3600
    drain_after_s: int = 5 * 3600 + 15 * 60
    final_save_after_s: int = 5 * 3600 + 20 * 60
    hard_return_s: int = 5 * 3600 + 30 * 60
    certification_status: str = 'NOT_CERTIFIED'

    def canonical_json(self) -> str:
        return json.dumps(asdict(self), sort_keys=True, separators=(',', ':'))

    def config_hash(self) -> str:
        return hashlib.sha256(self.canonical_json().encode()).hexdigest()

CONFIG = RunConfig()
print(CONFIG)
print('config_hash:', CONFIG.config_hash())

## Atomic file utilities

In [ ]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest()

def atomic_write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp-' + uuid.uuid4().hex)
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, sort_keys=True)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def fsync_directory(path: Path) -> None:
    try:
        fd = os.open(path, os.O_RDONLY)
    except OSError:
        return
    try:
        os.fsync(fd)
    finally:
        os.close(fd)

## Session guard

In [ ]:
@dataclass
class SessionGuard:
    config: RunConfig
    started_monotonic: float = field(default_factory=time.monotonic)
    last_checkpoint_monotonic: float = field(default_factory=time.monotonic)

    def elapsed_s(self) -> float:
        return time.monotonic() - self.started_monotonic

    def remaining_s(self) -> float:
        return max(0.0, self.config.hard_return_s - self.elapsed_s())

    def state(self) -> Literal['RUN', 'NO_LONG_TASK', 'DRAIN', 'FINAL_SAVE', 'RETURN']:
        elapsed = self.elapsed_s()
        if elapsed >= self.config.hard_return_s:
            return 'RETURN'
        if elapsed >= self.config.final_save_after_s:
            return 'FINAL_SAVE'
        if elapsed >= self.config.drain_after_s:
            return 'DRAIN'
        if elapsed >= self.config.no_long_task_after_s:
            return 'NO_LONG_TASK'
        return 'RUN'

    def checkpoint_due(self) -> bool:
        return time.monotonic() - self.last_checkpoint_monotonic >= self.config.checkpoint_interval_s

    def mark_checkpoint(self) -> None:
        self.last_checkpoint_monotonic = time.monotonic()

    def may_start(self, predicted_s: float, reserve_s: float = 12 * 60) -> bool:
        if self.state() in {'DRAIN', 'FINAL_SAVE', 'RETURN'}:
            return False
        if predicted_s > self.config.max_work_item_s:
            return False
        return self.remaining_s() >= 1.3 * predicted_s + reserve_s

## Persistent work queue

In [ ]:
WorkStatus = Literal['pending', 'running', 'done', 'failed']

@dataclass
class WorkItem:
    item_id: str
    phase: str
    payload: dict[str, Any]
    status: WorkStatus = 'pending'
    attempts: int = 0
    predicted_s: float = 60.0
    result_relpath: str | None = None
    result_sha256: str | None = None
    last_error: str | None = None

@dataclass
class WorkQueue:
    items: dict[str, WorkItem] = field(default_factory=dict)

    @staticmethod
    def make_id(phase: str, payload: dict[str, Any]) -> str:
        raw = json.dumps({'phase': phase, 'payload': payload}, sort_keys=True, separators=(',', ':')).encode()
        return hashlib.sha256(raw).hexdigest()

    def add(self, phase: str, payload: dict[str, Any], predicted_s: float = 60.0) -> str:
        item_id = self.make_id(phase, payload)
        self.items.setdefault(item_id, WorkItem(item_id, phase, payload, predicted_s=predicted_s))
        return item_id

    def next_pending(self) -> WorkItem | None:
        for item_id in sorted(self.items):
            if self.items[item_id].status == 'pending':
                return self.items[item_id]
        return None

    def recover_interrupted(self, run_root: Path) -> None:
        for item in self.items.values():
            if item.status != 'running':
                continue
            if item.result_relpath and item.result_sha256:
                result = run_root / item.result_relpath
                if result.exists() and sha256_file(result) == item.result_sha256:
                    item.status = 'done'
                    continue
            item.status = 'pending' 

## Checkpoint manager

In [ ]:
@dataclass
class RunState:
    run_id: str
    config_hash: str
    created_at: str
    updated_at: str
    phase: str = 'BOOTSTRAP'
    subphase: str = ''
    rg_step: int = 0
    checkpoint_index: int = 0
    certification_status: str = 'NOT_CERTIFIED'
    bounds: dict[str, str] = field(default_factory=dict)
    notes: list[str] = field(default_factory=list)

class CheckpointError(RuntimeError):
    pass

class CheckpointManager:
    def __init__(self, run_root: Path, config: RunConfig):
        self.run_root = run_root
        self.config = config
        self.ckpt_root = run_root / 'checkpoints'
        self.ckpt_root.mkdir(parents=True, exist_ok=True)

    def _rng_payload(self) -> dict[str, Any]:
        payload: dict[str, Any] = {
            'python': random.getstate(),
            'numpy': np.random.get_state(),
        }
        if torch is not None:
            payload['torch_cpu'] = torch.get_rng_state()
            if torch.cuda.is_available():
                payload['torch_cuda'] = torch.cuda.get_rng_state_all()
        return payload

    def _restore_rng(self, payload: dict[str, Any]) -> None:
        random.setstate(payload['python'])
        np.random.set_state(payload['numpy'])
        if torch is not None and 'torch_cpu' in payload:
            torch.set_rng_state(payload['torch_cpu'])
            if torch.cuda.is_available() and 'torch_cuda' in payload:
                torch.cuda.set_rng_state_all(payload['torch_cuda'])

    def save(self, state: RunState, queue: WorkQueue, tensors: dict[str, Any] | None = None) -> Path:
        state.updated_at = utc_now()
        previous_idx = state.checkpoint_index
        existing_indices = []
        for existing in self.ckpt_root.glob('ckpt_*'):
            try:
                existing_indices.append(int(existing.name.split('_', 1)[1]))
            except (IndexError, ValueError):
                continue
        highest_existing = max(existing_indices, default=0)
        next_idx = max(previous_idx, highest_existing) + 1
        state.checkpoint_index = next_idx
        tmp = self.ckpt_root / ('.tmp-' + uuid.uuid4().hex)
        final = self.ckpt_root / f'ckpt_{next_idx:06d}'
        tmp.mkdir(parents=True, exist_ok=False)

        atomic_write_json(tmp / 'state.json', asdict(state))
        atomic_write_json(tmp / 'work_queue.json', {'items': {k: asdict(v) for k, v in queue.items.items()}})
        atomic_write_json(tmp / 'meta.json', {
            'saved_at': utc_now(),
            'config': asdict(self.config),
            'config_hash': self.config.config_hash(),
            'python': sys.version,
            'platform': platform.platform(),
            'torch': getattr(torch, '__version__', None),
            'cuda_available': bool(torch and torch.cuda.is_available()),
            'gpu': torch.cuda.get_device_name(0) if torch and torch.cuda.is_available() else None,
        })

        if torch is not None:
            torch.save(self._rng_payload(), tmp / 'rng_state.pt')
            tensor_dir = tmp / 'tensors'
            tensor_dir.mkdir()
            for name, tensor in (tensors or {}).items():
                torch.save(tensor.detach().cpu(), tensor_dir / (name.replace('/', '__') + '.pt'))

        hashes = {}
        for path in sorted(tmp.rglob('*')):
            if path.is_file() and path.name not in {'hashes.json', 'COMMITTED'}:
                hashes[str(path.relative_to(tmp))] = sha256_file(path)
        atomic_write_json(tmp / 'hashes.json', hashes)
        (tmp / 'COMMITTED').write_text(utc_now(), encoding='utf-8')
        fsync_directory(tmp)
        os.replace(tmp, final)
        fsync_directory(self.ckpt_root)
        atomic_write_json(self.ckpt_root / 'LATEST.json', {'checkpoint': final.name, 'saved_at': utc_now()})
        try:
            self.verify(final)
        except Exception:
            state.checkpoint_index = previous_idx
            raise
        return final

    def verify(self, path: Path) -> None:
        if not (path / 'COMMITTED').exists():
            raise CheckpointError(f'Missing COMMITTED marker: {path}')
        hashes = json.loads((path / 'hashes.json').read_text(encoding='utf-8'))
        for rel, expected in hashes.items():
            file_path = path / rel
            if not file_path.exists() or sha256_file(file_path) != expected:
                raise CheckpointError(f'Invalid checkpoint file: {file_path}')

    def load_latest(self):
        candidates = sorted((p for p in self.ckpt_root.glob('ckpt_*') if p.is_dir()), reverse=True)
        for path in candidates:
            try:
                self.verify(path)
                state = RunState(**json.loads((path / 'state.json').read_text(encoding='utf-8')))
                if state.config_hash != self.config.config_hash():
                    raise CheckpointError('Config hash mismatch.')
                q_raw = json.loads((path / 'work_queue.json').read_text(encoding='utf-8'))
                queue = WorkQueue(items={k: WorkItem(**v) for k, v in q_raw['items'].items()})
                if torch is not None and (path / 'rng_state.pt').exists():
                    rng = torch.load(path / 'rng_state.pt', map_location='cpu', weights_only=False)
                    self._restore_rng(rng)
                return state, queue, path
            except Exception as exc:
                print(f'Skipping invalid checkpoint {path.name}: {exc}')
        return None

## Orchestrator scaffold

In [ ]:
class Orchestrator:
    def __init__(self, run_root: Path, config: RunConfig, state: RunState, queue: WorkQueue):
        self.run_root = run_root
        self.config = config
        self.state = state
        self.queue = queue
        self.guard = SessionGuard(config)
        self.checkpoints = CheckpointManager(run_root, config)
        self.tensors: dict[str, Any] = {}

    def checkpoint(self, reason: str) -> Path:
        self.state.notes.append(f'{utc_now()} checkpoint: {reason}')
        path = self.checkpoints.save(self.state, self.queue, self.tensors)
        self.guard.mark_checkpoint()
        print('Checkpoint committed:', path)
        return path

    def _execute_dummy(self, item: WorkItem) -> Path:
        item.attempts += 1
        item.status = 'running'
        self.checkpoint('before ' + item.item_id[:12])
        result_dir = self.run_root / 'artifacts' / item.item_id
        tmp = result_dir.with_name(result_dir.name + '.tmp-' + uuid.uuid4().hex)
        tmp.mkdir(parents=True, exist_ok=True)
        size = int(item.payload.get('size', 128))
        steps = int(item.payload.get('steps', 2))
        if torch is None:
            value = np.eye(size, dtype=np.float64)
            for _ in range(steps):
                value = value @ value.T / max(1.0, np.linalg.norm(value))
            result_file = tmp / 'result.npy'
            np.save(result_file, value[:8, :8])
        else:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
            x = torch.eye(size, dtype=torch.float64, device=device)
            for _ in range(steps):
                x = x @ x.T
                x = x / torch.linalg.norm(x)
            result_file = tmp / 'result.pt'
            torch.save(x[:8, :8].cpu(), result_file)
        result_dir.parent.mkdir(parents=True, exist_ok=True)
        result_name = result_file.name
        os.replace(tmp, result_dir)
        committed_result = result_dir / result_name
        item.result_relpath = str(committed_result.relative_to(self.run_root))
        item.result_sha256 = sha256_file(committed_result)
        item.status = 'done'
        self.checkpoint('after ' + item.item_id[:12])
        return result_file

    def run_until_checkpoint(self) -> dict[str, Any]:
        print('Session started:', utc_now())
        print('Run ID:', self.state.run_id)
        print('Certification status:', self.state.certification_status)
        while True:
            state = self.guard.state()
            if state in {'FINAL_SAVE', 'RETURN'}:
                self.checkpoint('session state ' + state)
                break
            if state == 'DRAIN':
                self.checkpoint('drain mode')
                break
            if self.guard.checkpoint_due():
                self.checkpoint('periodic')
            item = self.queue.next_pending()
            if item is None:
                self.state.phase = 'M0_COMPLETE'
                self.checkpoint('queue complete')
                break
            if not self.guard.may_start(item.predicted_s):
                self.checkpoint('insufficient time for next item')
                break
            try:
                if item.phase == 'DUMMY':
                    self._execute_dummy(item)
                else:
                    raise NotImplementedError(f'Phase {item.phase} is not implemented in M0.')
            except KeyboardInterrupt:
                item.status = 'pending'
                self.checkpoint('KeyboardInterrupt')
                raise
            except Exception as exc:
                item.status = 'failed'
                item.last_error = repr(exc)
                self.checkpoint('item failed')
                raise
        summary = {
            'run_id': self.state.run_id,
            'phase': self.state.phase,
            'checkpoint_index': self.state.checkpoint_index,
            'certification_status': self.state.certification_status,
            'elapsed_s': self.guard.elapsed_s(),
            'next_action': 'Restart, mount the same persistent root, rerun bootstrap, resume, and call run_until_checkpoint() again.',
        }
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        return summary

## Create or resume a run

In [ ]:
def create_or_resume(persistent_root: Path, config: RunConfig, run_id: str | None = None) -> Orchestrator:
    runs_root = persistent_root / 'runs'
    runs_root.mkdir(parents=True, exist_ok=True)
    if run_id is None:
        candidates = sorted(p for p in runs_root.iterdir() if p.is_dir())
        run_root = candidates[-1] if candidates else None
    else:
        run_root = runs_root / run_id
    if run_root is not None and run_root.exists():
        loaded = CheckpointManager(run_root, config).load_latest()
        if loaded is not None:
            state, queue, ckpt = loaded
            queue.recover_interrupted(run_root)
            print('Resumed from:', ckpt)
            return Orchestrator(run_root, config, state, queue)
    new_run_id = run_id or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + config.config_hash()[:8]
    run_root = runs_root / new_run_id
    run_root.mkdir(parents=True, exist_ok=False)
    atomic_write_json(run_root / 'run_config.json', asdict(config))
    state = RunState(new_run_id, config.config_hash(), utc_now(), utc_now())
    queue = WorkQueue()
    for idx in range(4):
        queue.add('DUMMY', {'index': idx, 'size': 128, 'steps': 2}, predicted_s=30)
    orchestrator = Orchestrator(run_root, config, state, queue)
    orchestrator.checkpoint('initial')
    return orchestrator

# Main use:
# PERSIST_ROOT = resolve_persistent_root()
# orchestrator = create_or_resume(PERSIST_ROOT, CONFIG)
# orchestrator.run_until_checkpoint()

## Required M0 tests

Codex must implement automated tests for:

1. atomic save/load;
2. SHA-256 verification;
3. corrupt-latest fallback;
4. RNG restoration;
5. interrupted work-item recovery;
6. short simulated session budget;
7. fresh-process resume;
8. CPU-only operation;
9. optional CUDA dummy workload;
10. `certification_status` remaining `NOT_CERTIFIED`.

Do not start mathematical RG implementation until these pass.

## Mathematical phases still intentionally absent

- representation/Wigner cache;
- fusion enumeration;
- 4D armillary tensor;
- matrix-free Triad-ATRG;
- forward source derivatives;
- analytic tails;
- deterministic RSVD residual bounds;
- interval influence matrix;
- Collatz–Wielandt certificate.

Their implementation order and acceptance tests are specified in the design document.

## Next session

```python
PERSIST_ROOT = resolve_persistent_root()
orchestrator = create_or_resume(PERSIST_ROOT, CONFIG)
orchestrator.run_until_checkpoint()
```

The loader scans checkpoints newest-first, verifies hashes, and falls back to the
previous valid checkpoint if necessary.